# IMC25 – Single submission notebook

This notebook is intended to be **the one notebook** you run to produce a `submission.csv`.

It follows this flow:

1. **Train** a retrieval embedder using `train_labels.csv`
2. **Embed** all test images → `cache_retrieval/`
3. **Cluster** embeddings → `cache/clusters_retrieval.csv`
4. **Run SfM** per cluster (COLMAP) → sparse poses under `outputs_submission/<dataset>_<scene>/sparse/...`
5. **Export** a competition-format `submission.csv`

Notes:
- This notebook is self-contained (no `src/` / `scripts/` upload needed).
- Kaggle (no internet): add DINOv2 via “Add Model”/Dataset and set `MODEL_DIR` (or env `IMC25_MODEL_DIR`) to either a Transformers folder (`config.json` + weights) or a weights folder (`.pth/.pt`) for timm fallback (optional: `IMC25_TIMM_WEIGHTS`).
- Data root auto-detects `./data/` or Kaggle `/kaggle/input/...`; set `IMC25_DATA_ROOT` to override.
- The SfM step requires a `colmap` binary in `PATH` (Kaggle does not support Docker). If missing, SfM is skipped and poses are `nan`. To run SfM offline, set env `IMC25_COLMAP_BIN` to a `colmap` executable you added as a Kaggle Dataset.


In [ ]:
from __future__ import annotations

import json
import os
import struct
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd


WORK_ROOT = Path.cwd().resolve()

IS_KAGGLE = Path("/kaggle").exists()
if IS_KAGGLE:
    os.environ.setdefault("HF_HUB_OFFLINE", "1")
    os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
    os.environ.setdefault("HF_HOME", str(WORK_ROOT / "hf_home"))
    os.environ.setdefault("QT_QPA_PLATFORM", "offscreen")
    os.environ.setdefault("LIBGL_ALWAYS_SOFTWARE", "1")

    _ki = Path("/kaggle/input")
    if _ki.exists():
        print("kaggle input dirs:", sorted([p.name for p in _ki.iterdir() if p.is_dir()]))



def find_data_root() -> Path:
    """Return folder containing train_labels.csv + train/ + test/."""
    env = os.environ.get("IMC25_DATA_ROOT")
    if env:
        p = Path(env).expanduser().resolve()
        if not (p / "train_labels.csv").exists():
            raise FileNotFoundError(f"IMC25_DATA_ROOT does not contain train_labels.csv: {p}")
        return p

    local = WORK_ROOT / "data"
    if (local / "train_labels.csv").exists() and (local / "train").is_dir() and (local / "test").is_dir():
        return local.resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for cand in sorted([p for p in kaggle_input.iterdir() if p.is_dir()]):
            roots = [cand]
            try:
                roots += [d for d in cand.iterdir() if d.is_dir()]
            except Exception:
                pass
            for root in roots:
                if (root / "train_labels.csv").exists() and (root / "train").is_dir() and (root / "test").is_dir():
                    return root.resolve()

    raise RuntimeError(
        "Could not find IMC25 data root. Set IMC25_DATA_ROOT to the folder containing train_labels.csv + train/ + test/."
    )


DATA_ROOT = find_data_root()

print("work:", WORK_ROOT)
print("data:", DATA_ROOT)
print("HF_ENDPOINT:", os.environ.get("HF_ENDPOINT"))
print("python:", sys.version.split()[0])


## Paths + flags

- Kaggle mounts added datasets/models under `/kaggle/input/<name>/...`.
- For offline runs, set `MODEL_DIR` to the folder containing the Transformers files (`config.json` + weights).
- Set `IMC25_DATA_ROOT` / `IMC25_MODEL_DIR` env vars if you prefer not to edit cells.


In [ ]:
# Data
TRAIN_LABELS_CSV = DATA_ROOT / "train_labels.csv"
TRAIN_ROOT = DATA_ROOT / "train"
TEST_ROOT = DATA_ROOT / "test"

# Outputs (written to WORK_ROOT)
OUTPUT_ROOT = WORK_ROOT / "outputs_submission"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Retrieval model
MODEL_ID = "facebook/dinov2-small"  # Hugging Face repo id (offline requires local files)
HF_LOCAL_ONLY = Path("/kaggle").exists()  # Kaggle notebooks typically run without internet

# Option A (recommended for offline Kaggle): point to a local Transformers folder (contains config.json + weights)
MODEL_DIR = os.environ.get("IMC25_MODEL_DIR")  # e.g. /kaggle/input/<your-model>/.../

# Local HF cache dir (used for Transformers cache bookkeeping; model files can still live in /kaggle/input)
HF_CACHE_DIR = WORK_ROOT / "hf_cache"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _looks_like_transformers_model_dir(d: Path) -> bool:
    if not (d / "config.json").exists():
        return False

    direct = {"model.safetensors", "pytorch_model.bin", "pytorch_model.bin.index.json"}
    if any((d / n).exists() for n in direct):
        return True

    try:
        for p in d.iterdir():
            if not p.is_file():
                continue
            if p.suffix == ".safetensors":
                return True
            if p.name.startswith("pytorch_model") and p.suffix == ".bin":
                return True
    except Exception:
        return False

    return False


def _auto_find_model_dir(model_id: str) -> str | None:
    """Best-effort search for a local Transformers model folder under /kaggle/input."""
    root = Path("/kaggle/input")
    if not root.exists():
        return None

    tokens = [t for t in model_id.lower().replace("/", " ").replace("-", " ").split() if t]
    skip = {"train", "test", "data", "images", "image", "labels", "cache"}
    max_depth = 6

    candidates: list[Path] = []
    for top in sorted([p for p in root.iterdir() if p.is_dir()]):
        if _looks_like_transformers_model_dir(top):
            candidates.append(top)

        for dirpath, dirnames, filenames in os.walk(top):
            rel = Path(dirpath).relative_to(top)
            if len(rel.parts) > max_depth:
                dirnames[:] = []
                continue

            # Avoid walking huge image folders.
            dirnames[:] = [d for d in dirnames if d not in skip and not d.startswith(".")]

            if "config.json" not in filenames:
                continue

            d = Path(dirpath)
            if _looks_like_transformers_model_dir(d):
                candidates.append(d)

    uniq: list[Path] = []
    seen: set[Path] = set()
    for c in candidates:
        c = c.resolve()
        if c in seen:
            continue
        seen.add(c)
        uniq.append(c)

    def score(p: Path) -> tuple[int, int]:
        s = p.as_posix().lower()
        hits = sum(1 for t in tokens if t in s)
        return hits, -len(s)

    uniq.sort(key=score, reverse=True)
    return str(uniq[0]) if uniq else None


if MODEL_DIR:
    _p = Path(MODEL_DIR)
    if not _p.exists():
        print("[warn] MODEL_DIR does not exist; ignoring:", _p)
        MODEL_DIR = None
    elif not (_p / "config.json").exists():
        print(
            "[warn] MODEL_DIR missing config.json; Transformers load may fail (timm fallback will be attempted if it contains .pth/.pt weights):",
            _p,
        )

if HF_LOCAL_ONLY and not MODEL_DIR:
    _default = Path("/kaggle/input/dinov2/pytorch/small/1")
    if _default.exists():
        MODEL_DIR = str(_default)
        print("[info] using Kaggle model mount for DINOv2:", MODEL_DIR)

if HF_LOCAL_ONLY and not MODEL_DIR:
    _auto = _auto_find_model_dir(MODEL_ID)
    if _auto:
        MODEL_DIR = _auto
        print("[info] auto-detected MODEL_DIR:", MODEL_DIR)
    else:
        print("[warn] Could not auto-detect model files under /kaggle/input; set MODEL_DIR or IMC25_MODEL_DIR.")

MODEL_REF = MODEL_DIR if MODEL_DIR else MODEL_ID

# Optional fine-tuned checkpoint
TRAIN_OUTPUT_DIR = WORK_ROOT / "outputs" / "retrieval_finetune"
CHECKPOINT = TRAIN_OUTPUT_DIR / "best.pt"

# Pipeline flags
DO_TRAIN = False  # if False, will use MODEL_REF (pretrained) if CHECKPOINT is missing
DO_EMBED_TEST = True
DO_CLUSTER = True
DO_COLMAP_SFM = True

# Edge scorer Config
EDGE_SCORER = "cosine"  # "cosine" | "gbdt" | "mlp"

EDGE_MODEL = {
    "mlp": WORK_ROOT / "outputs" / "edge_models" / "edge_mlp.pt",
    "gbdt": WORK_ROOT / "outputs" / "edge_models" / "edge_gbdt.joblib",
}.get(EDGE_SCORER)

EDGE_TOPM = 5  # None or integer (determines how many top matches to keep)

# COLMAP settings
RUNNER = "local"  # Kaggle does not support Docker
MATCHER = "colmap"  # "colmap" only in this notebook
SPARSE_ONLY = True  # True = SfM only (faster); False = SfM + dense MVS

COLMAP_SIFT_USE_GPU = False  # Set True only if your COLMAP can create an OpenGL context

# CSV locations
CACHE_RETRIEVAL = WORK_ROOT / "cache_retrieval"
CLUSTERS_CSV = WORK_ROOT / "cache" / "clusters_retrieval.csv"
# Kaggle code-submission expects /kaggle/working/submission.csv
SUBMISSION_CSV = WORK_ROOT / "submission.csv"

assert TRAIN_LABELS_CSV.exists(), f"Missing {TRAIN_LABELS_CSV}"
assert TRAIN_ROOT.exists(), f"Missing {TRAIN_ROOT}"
assert TEST_ROOT.exists(), f"Missing {TEST_ROOT}"

print("model_ref:", MODEL_REF)
print("hf_local_only:", HF_LOCAL_ONLY)

SUBMISSION_CSV


PosixPath('/Users/luca/Documents/Uni/Tsinghua/Machine Learning/ImageMatchingChallenge2025/outputs_submission/submission.csv')

## Implementation (self-contained)

This section embeds the minimal code from `src/` and `scripts/` so the notebook can run standalone on Kaggle.


In [3]:
# Copied/adapted from src/imc25 + scripts so this notebook runs without extra uploads.
# Notes:
# - If a fine-tuned checkpoint is missing, we fall back to pretrained MODEL_ID embeddings.
# - If Hugging Face weights cannot be loaded (e.g., Kaggle internet disabled), we fall back to simple color-hist embeddings.

from dataclasses import dataclass
import random
import shutil
import time

import numpy as np

import joblib
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.neighbors import NearestNeighbors
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from torchvision import transforms as T


# COLMAP path resolution (Kaggle datasets may mount binaries without exec bit).
_COLMAP_BIN_CACHE: str | None = None


def _resolve_colmap_bin() -> str | None:
    global _COLMAP_BIN_CACHE
    if _COLMAP_BIN_CACHE is not None:
        return _COLMAP_BIN_CACHE

    env = os.environ.get("IMC25_COLMAP_BIN")
    candidates: list[Path] = []

    def _looks_like_executable(p: Path) -> bool:
        try:
            with p.open("rb") as f:
                head = f.read(4)
        except Exception:
            return False
        return head.startswith(b"\x7fELF") or head.startswith(b"#!")

    def add(p: Path) -> None:
        if p.exists() and p.is_file() and _looks_like_executable(p):
            candidates.append(p)

    if env:
        p = Path(env).expanduser()
        if p.is_file():
            add(p)
        elif p.is_dir():
            add(p / "colmap")
            add(p / "bin" / "colmap")
            add(p / "usr" / "bin" / "colmap")
            add(p / "usr" / "local" / "bin" / "colmap")

            max_depth = 5
            try:
                for dirpath, dirnames, filenames in os.walk(p):
                    rel = Path(dirpath).relative_to(p)
                    if len(rel.parts) >= max_depth:
                        dirnames[:] = []
                        continue
                    if "colmap" in filenames:
                        add(Path(dirpath) / "colmap")
            except Exception:
                pass

    which = shutil.which("colmap")
    if which:
        add(Path(which))

    # Kaggle: COLMAP might be provided via an added dataset/model under /kaggle/input.
    if not candidates:
        kaggle_input = Path("/kaggle/input")
        if kaggle_input.exists():
            skip = {"train", "test", "data", "images", "image", "labels", "cache"}
            max_depth = 6
            for top in sorted([d for d in kaggle_input.iterdir() if d.is_dir()]):
                try:
                    for dirpath, dirnames, filenames in os.walk(top):
                        rel = Path(dirpath).relative_to(top)
                        if len(rel.parts) >= max_depth:
                            dirnames[:] = []
                            continue
                        dirnames[:] = [d for d in dirnames if d not in skip and not d.startswith(".")]
                        if "colmap" in filenames:
                            add(Path(dirpath) / "colmap")
                except Exception:
                    continue

    if not candidates:
        _COLMAP_BIN_CACHE = None
        return None

    def _is_elf(p: Path) -> bool:
        try:
            with p.open("rb") as f:
                return f.read(4) == b"\x7fELF"
        except Exception:
            return False

    # Prefer ELF binaries; then prefer the shortest path.
    candidates = sorted({c.resolve() for c in candidates}, key=lambda x: (0 if _is_elf(x) else 1, len(str(x)), str(x)))
    chosen = candidates[0]

    chosen_src = chosen

    # If the binary isn't executable (common for Kaggle inputs), copy it into WORK_ROOT and chmod.
    if not os.access(str(chosen), os.X_OK):
        try:
            work_root = globals().get("WORK_ROOT")
            work_root = Path(work_root) if work_root else Path.cwd().resolve()
            out = work_root / "bin" / "colmap"
            out.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(chosen, out)
            out.chmod(0o755)
            chosen = out
        except Exception:
            pass

    def _prepend_env_path(var: str, value: Path) -> None:
        val = str(value)
        cur = os.environ.get(var, "")
        parts = [p for p in cur.split(":") if p]
        if val in parts:
            return
        os.environ[var] = val + (":" + cur if cur else "")

    def _infer_lib_dirs(colmap_bin: Path) -> list[Path]:
        bases: list[Path] = []
        # Common portable layout: <root>/bin/colmap and <root>/lib/...
        if colmap_bin.parent.name == "bin":
            bases.append(colmap_bin.parent.parent)
        bases.append(colmap_bin.parent)
        bases.extend(list(colmap_bin.parents)[:5])

        uniq: list[Path] = []
        seen: set[Path] = set()
        for b in bases:
            br = b.resolve()
            if br in seen:
                continue
            seen.add(br)
            uniq.append(br)

        out: list[Path] = []
        for b in uniq:
            for dname in ("lib", "lib64"):
                d = b / dname
                if d.is_dir():
                    out.append(d.resolve())
        # Preserve order, dedupe.
        out_uniq: list[Path] = []
        out_seen: set[Path] = set()
        for d in out:
            if d in out_seen:
                continue
            out_seen.add(d)
            out_uniq.append(d)
        return out_uniq

    # Mimic the portable-colmap setup pattern: ensure PATH and LD_LIBRARY_PATH include bundled libs.
    _prepend_env_path("PATH", Path(chosen).parent)
    for lib_dir in _infer_lib_dirs(Path(chosen_src)):
        _prepend_env_path("LD_LIBRARY_PATH", lib_dir)

    _COLMAP_BIN_CACHE = str(chosen)
    return _COLMAP_BIN_CACHE


def run_cmd(cmd: list[str], *, cwd: Path | None = None) -> None:
    from collections import deque

    cmd = [str(x) for x in cmd]

    is_colmap = bool(cmd) and (cmd[0] == "colmap" or Path(cmd[0]).name == "colmap")
    if is_colmap and cmd[0] == "colmap":
        resolved = _resolve_colmap_bin()
        if not resolved:
            raise FileNotFoundError(
                "`colmap` not found. Set IMC25_COLMAP_BIN to the colmap executable (or a folder containing it)."
            )
        cmd[0] = str(resolved)

    is_colmap = bool(cmd) and Path(cmd[0]).name == "colmap"
    if is_colmap:
        # Some COLMAP builds require an OpenGL context even for CLI SIFT.
        if not os.environ.get("DISPLAY") and shutil.which("xvfb-run") is not None:
            cmd = ["xvfb-run", "-a", *cmd]

    print("+", " ".join(cmd), flush=True)

    try:
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd) if cwd else None,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
    except FileNotFoundError as e:
        raise FileNotFoundError(f"Command not found: {cmd[0]}") from e
    tail = deque(maxlen=200)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        tail.append(line.rstrip("\n"))
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd, output="\n".join(tail))


# -------------------- Retrieval model + transforms --------------------


@dataclass(frozen=True)
class RetrievalModelConfig:
    model_id: str
    embed_dim: int = 256
    use_fp16: bool = True
    local_files_only: bool = False
    cache_dir: str | None = None


def _load_backbone(model_id: str, *, local_files_only: bool = False, cache_dir=None):
    from transformers import AutoConfig, AutoModel

    try:
        from transformers import CLIPVisionModel
    except Exception:  # pragma: no cover
        CLIPVisionModel = None  # type: ignore[assignment]

    cfg = AutoConfig.from_pretrained(
        model_id,
        local_files_only=bool(local_files_only),
        cache_dir=str(cache_dir) if cache_dir else None,
    )
    if getattr(cfg, "model_type", "") == "clip" and CLIPVisionModel is not None:
        return CLIPVisionModel.from_pretrained(
            model_id,
            local_files_only=bool(local_files_only),
            cache_dir=str(cache_dir) if cache_dir else None,
        )
    return AutoModel.from_pretrained(
        model_id,
        local_files_only=bool(local_files_only),
        cache_dir=str(cache_dir) if cache_dir else None,
    )


def _backbone_dim(backbone) -> int:
    for attr in ("hidden_size", "vision_embed_dim", "projection_dim", "embedding_size"):
        v = getattr(backbone.config, attr, None)
        if isinstance(v, int) and v > 0:
            return int(v)
    raise ValueError(f"Could not determine embedding dimension for backbone: {type(backbone)}")


def _pool_output(out) -> torch.Tensor:
    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        return out.pooler_output
    if hasattr(out, "last_hidden_state") and out.last_hidden_state is not None:
        return out.last_hidden_state[:, 0]
    raise ValueError("Backbone output does not expose pooler_output or last_hidden_state")


class RetrievalEmbedder(nn.Module):
    def __init__(self, cfg: RetrievalModelConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.backbone = _load_backbone(
            cfg.model_id,
            local_files_only=bool(cfg.local_files_only),
            cache_dir=cfg.cache_dir,
        )
        dim = _backbone_dim(self.backbone)
        if int(cfg.embed_dim) == int(dim):
            self.proj = nn.Identity()
        else:
            self.proj = nn.Sequential(
                nn.Linear(dim, int(cfg.embed_dim)),
                nn.GELU(),
                nn.Linear(int(cfg.embed_dim), int(cfg.embed_dim)),
            )

    @property
    def embed_dim(self) -> int:
        return int(self.cfg.embed_dim)

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        out = self.backbone(pixel_values=pixel_values)
        x = _pool_output(out)
        z = self.proj(x)
        z = F.normalize(z, p=2, dim=1)
        return z


@dataclass(frozen=True)
class TransformSpec:
    image_size: tuple[int, int]
    mean: tuple[float, float, float]
    std: tuple[float, float, float]


def _infer_size(size) -> tuple[int, int]:
    if isinstance(size, int):
        return int(size), int(size)
    if isinstance(size, dict):
        if "height" in size and "width" in size:
            return int(size["height"]), int(size["width"])
        if "shortest_edge" in size:
            s = int(size["shortest_edge"])
            return s, s
    raise ValueError(f"Unsupported processor.size: {size}")


def load_transform_spec(
    *,
    model_id: str,
    hf_endpoint: str | None,
    local_files_only: bool = False,
    cache_dir=None,
) -> TransformSpec:
    if hf_endpoint:
        os.environ["HF_ENDPOINT"] = str(hf_endpoint)

    try:
        from transformers import AutoImageProcessor

        proc = AutoImageProcessor.from_pretrained(
            model_id,
            local_files_only=bool(local_files_only),
            cache_dir=str(cache_dir) if cache_dir else None,
        )
        size = _infer_size(getattr(proc, "size", {"shortest_edge": 224}))
        mean = tuple(float(x) for x in getattr(proc, "image_mean", (0.485, 0.456, 0.406)))
        std = tuple(float(x) for x in getattr(proc, "image_std", (0.229, 0.224, 0.225)))
        if len(mean) != 3 or len(std) != 3:
            raise ValueError(f"Unexpected mean/std from processor: mean={mean} std={std}")
        return TransformSpec(image_size=size, mean=mean, std=std)
    except Exception as e:
        print(
            f"[warn] AutoImageProcessor unavailable for {model_id}: {type(e).__name__}: {e}; using default transforms",
            flush=True,
        )
        return TransformSpec(
            image_size=(224, 224),
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        )


def make_train_transform(spec: TransformSpec):
    h, w = spec.image_size
    return T.Compose(
        [
            T.RandomResizedCrop((h, w), scale=(0.6, 1.0), ratio=(0.75, 1.3333)),
            T.RandomHorizontalFlip(p=0.5),
            T.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.35, hue=0.08),
            T.RandomGrayscale(p=0.15),
            T.ToTensor(),
            T.Normalize(mean=spec.mean, std=spec.std),
        ]
    )


def make_eval_transform(spec: TransformSpec):
    h, w = spec.image_size
    resize_to = int(round(max(h, w) / 0.875))
    return T.Compose(
        [
            T.Resize(resize_to, interpolation=T.InterpolationMode.BICUBIC),
            T.CenterCrop((h, w)),
            T.ToTensor(),
            T.Normalize(mean=spec.mean, std=spec.std),
        ]
    )


def _default_transform_spec() -> TransformSpec:
    return TransformSpec(
        image_size=(224, 224),
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )


def _find_weights_file(root: Path) -> Path | None:
    exts = {".pth", ".pt", ".bin", ".safetensors"}
    if root.exists() and root.is_file():
        return root.resolve()
    if not (root.exists() and root.is_dir()):
        return None

    candidates: list[Path] = []
    max_depth = 6
    for dirpath, dirnames, filenames in os.walk(root):
        rel = Path(dirpath).relative_to(root)
        if len(rel.parts) > max_depth:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        for fn in filenames:
            p = Path(dirpath) / fn
            if p.suffix.lower() in exts:
                candidates.append(p)

    if not candidates:
        return None

    def size(p: Path) -> int:
        try:
            return int(p.stat().st_size)
        except Exception:
            return -1

    candidates.sort(key=size, reverse=True)
    return candidates[0].resolve()


def _guess_timm_dinov2_name(model_ref: str, weights: Path | None) -> str | None:
    env = os.environ.get("IMC25_TIMM_MODEL")
    if env:
        return str(env)

    s = str(model_ref).lower()
    if weights is not None:
        s = s + " " + weights.name.lower()

    if "dinov2" not in s:
        return None

    reg4 = "reg4" in s
    if "vits14" in s or "small" in s:
        size = "small"
    elif "vitb14" in s or "base" in s:
        size = "base"
    elif "vitl14" in s or "large" in s:
        size = "large"
    elif "vitg14" in s or "giant" in s:
        size = "giant"
    else:
        size = "small"

    if reg4:
        return f"vit_{size}_patch14_reg4_dinov2"
    return f"vit_{size}_patch14_dinov2"


def _load_timm_dinov2_embedder(
    *,
    model_ref: str,
    device: torch.device,
) -> tuple[nn.Module, TransformSpec, int, dict]:
    try:
        import timm
    except Exception as e:
        raise ImportError("timm is required for the DINOv2 timm fallback.") from e

    weights: Path | None = None
    env_w = os.environ.get("IMC25_TIMM_WEIGHTS")
    if env_w:
        p = Path(env_w).expanduser()
        if p.exists():
            weights = p.resolve()

    if weights is None:
        p = Path(model_ref).expanduser()
        weights = _find_weights_file(p) if p.exists() else None

    if weights is None:
        raise FileNotFoundError(
            "Could not find local DINOv2 weights for timm fallback. "
            "Set MODEL_DIR to a folder containing a .pth/.pt, or set IMC25_TIMM_WEIGHTS."
        )

    model_name = _guess_timm_dinov2_name(model_ref, weights)
    if not model_name:
        raise ValueError(f"Not a DINOv2 reference: {model_ref}")

    dynamic = True
    try:
        backbone = timm.create_model(model_name, pretrained=False, dynamic_img_size=True)
    except TypeError:
        backbone = timm.create_model(model_name, pretrained=False)
        dynamic = False

    ckpt = torch.load(weights, map_location="cpu")
    state = ckpt
    if isinstance(ckpt, dict):
        if "state_dict" in ckpt and isinstance(ckpt["state_dict"], dict):
            state = ckpt["state_dict"]
        elif "model" in ckpt and isinstance(ckpt["model"], dict):
            state = ckpt["model"]
    if not isinstance(state, dict):
        raise ValueError(f"Unexpected checkpoint format: {type(ckpt)}")

    cleaned: dict[str, torch.Tensor] = {}
    for k, v in state.items():
        kk = str(k)
        for pref in ("module.", "backbone.", "model."):
            if kk.startswith(pref):
                kk = kk[len(pref) :]
        cleaned[kk] = v

    missing, unexpected = backbone.load_state_dict(cleaned, strict=False)
    backbone.eval().to(device)

    spec = _default_transform_spec()
    if not dynamic:
        cfg = getattr(backbone, "default_cfg", {}) or {}
        inp = cfg.get("input_size", None)
        if isinstance(inp, (tuple, list)) and len(inp) == 3:
            spec = TransformSpec(
                image_size=(int(inp[1]), int(inp[2])),
                mean=tuple(float(x) for x in cfg.get("mean", spec.mean)),
                std=tuple(float(x) for x in cfg.get("std", spec.std)),
            )

    class _TimmEmbedder(nn.Module):
        def __init__(self, backbone) -> None:
            super().__init__()
            self.backbone = backbone
            self._dim = int(getattr(backbone, "num_features", 0) or getattr(backbone, "embed_dim", 0) or 0)
            if self._dim <= 0:
                raise ValueError("Could not determine timm backbone embedding dimension")

        @property
        def embed_dim(self) -> int:
            return int(self._dim)

        def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
            if hasattr(self.backbone, "forward_features"):
                x = self.backbone.forward_features(pixel_values)
            else:
                x = self.backbone(pixel_values)
            if isinstance(x, (tuple, list)):
                x = x[0]
            if not isinstance(x, torch.Tensor):
                raise ValueError(f"Unexpected timm output type: {type(x)}")
            if x.ndim == 3:
                x = x[:, 0]
            elif x.ndim == 4:
                x = x.mean(dim=(2, 3))
            z = F.normalize(x.float(), p=2, dim=1)
            return z

    model = _TimmEmbedder(backbone).to(device)
    meta = {
        "backend": "timm",
        "timm_model": model_name,
        "weights": str(weights),
        "dynamic_img_size": bool(dynamic),
        "missing_keys": int(len(missing)),
        "unexpected_keys": int(len(unexpected)),
        "image_size": list(spec.image_size),
    }
    return model, spec, int(model.embed_dim), meta


# -------------------- Retrieval train --------------------


@dataclass(frozen=True)
class RetrievalSample:
    path: Path
    dataset: str
    scene: str
    image_id: str
    label: int


class RetrievalImageDataset(Dataset[tuple[torch.Tensor, int]]):
    def __init__(self, samples: list[RetrievalSample], *, transform) -> None:
        self.samples = samples
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        s = self.samples[int(idx)]
        with Image.open(s.path) as im:
            im = im.convert("RGB")
            x = self.transform(im)
        return x, int(s.label)


_IMG_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}


def _list_images(dataset_dir: Path) -> list[Path]:
    out: list[Path] = []
    for p in sorted(dataset_dir.iterdir()):
        if not p.is_file():
            continue
        if p.suffix.lower() in _IMG_EXTS:
            out.append(p)
    return out


def build_train_samples(
    *,
    data_root: Path,
    labels_csv: Path,
    datasets: set[str] | None = None,
    include_outliers: bool = True,
) -> tuple[list[RetrievalSample], dict[int, str]]:
    df = pd.read_csv(labels_csv)
    required = {"dataset", "scene", "image"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{labels_csv} missing columns: {sorted(missing)}")

    df = df.copy()
    df["dataset"] = df["dataset"].astype(str)
    df["scene"] = df["scene"].astype(str)
    df["image"] = df["image"].astype(str)

    if datasets is not None:
        df = df[df["dataset"].isin(sorted(datasets))].reset_index(drop=True)

    label_name_to_id: dict[str, int] = {}
    id_to_name: dict[int, str] = {}

    def get_label_id(name: str) -> int:
        if name in label_name_to_id:
            return label_name_to_id[name]
        idx = len(label_name_to_id)
        label_name_to_id[name] = idx
        id_to_name[idx] = name
        return idx

    samples: list[RetrievalSample] = []
    labeled_by_dataset: dict[str, set[str]] = {}
    for row in df.itertuples(index=False):
        dataset = str(row.dataset)
        scene = str(row.scene)
        img_name = str(row.image)
        path = (data_root / dataset / img_name).resolve()
        if not path.exists():
            raise FileNotFoundError(path)
        image_id = Path(img_name).stem
        labeled_by_dataset.setdefault(dataset, set()).add(image_id)
        label = get_label_id(f"{dataset}/{scene}")
        samples.append(
            RetrievalSample(
                path=path,
                dataset=dataset,
                scene=scene,
                image_id=image_id,
                label=label,
            )
        )

    if include_outliers:
        for dataset_dir in sorted([p for p in data_root.iterdir() if p.is_dir()]):
            dataset = dataset_dir.name
            if datasets is not None and dataset not in datasets:
                continue
            labeled = labeled_by_dataset.get(dataset, set())
            for p in _list_images(dataset_dir):
                image_id = p.stem
                if image_id in labeled:
                    continue
                label = get_label_id(f"{dataset}/outlier/{image_id}")
                samples.append(
                    RetrievalSample(
                        path=p.resolve(),
                        dataset=dataset,
                        scene="outlier",
                        image_id=image_id,
                        label=label,
                    )
                )

    return samples, id_to_name


def split_train_val(
    samples: list[RetrievalSample],
    *,
    val_ratio: float,
    seed: int,
) -> tuple[list[RetrievalSample], list[RetrievalSample]]:
    if not (0.0 <= val_ratio < 1.0):
        raise ValueError("val_ratio must be in [0,1)")

    by_label: dict[int, list[int]] = {}
    for i, s in enumerate(samples):
        by_label.setdefault(int(s.label), []).append(i)

    rng = random.Random(int(seed))
    train_idx: set[int] = set()
    val_idx: set[int] = set()

    for _, idxs in by_label.items():
        if len(idxs) <= 1:
            train_idx.update(idxs)
            continue
        idxs = idxs[:]
        rng.shuffle(idxs)
        n_val = int(round(len(idxs) * float(val_ratio)))
        n_val = max(1, min(n_val, len(idxs) - 1))
        val_idx.update(idxs[:n_val])
        train_idx.update(idxs[n_val:])

    train = [samples[i] for i in sorted(train_idx)]
    val = [samples[i] for i in sorted(val_idx)]
    return train, val


def supervised_contrastive_loss(
    z: torch.Tensor,
    labels: torch.Tensor,
    *,
    temperature: float = 0.07,
) -> torch.Tensor:
    if z.ndim != 2:
        raise ValueError(f"Expected z (B,D), got {tuple(z.shape)}")
    if labels.ndim != 1:
        raise ValueError(f"Expected labels (B,), got {tuple(labels.shape)}")
    if z.shape[0] != labels.shape[0]:
        raise ValueError("Batch size mismatch between z and labels")

    bsz = int(z.shape[0])
    if bsz <= 1:
        return z.new_tensor(0.0)

    z = F.normalize(z, p=2, dim=1)
    labels = labels.to(device=z.device)

    logits = (z @ z.T) / float(temperature)
    logits = logits - logits.max(dim=1, keepdim=True).values.detach()

    mask = labels.view(-1, 1).eq(labels.view(1, -1)).to(dtype=torch.float32)
    mask.fill_diagonal_(0.0)

    logits_mask = torch.ones_like(mask)
    logits_mask.fill_diagonal_(0.0)

    exp_logits = torch.exp(logits) * logits_mask
    log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-12)

    pos = mask.sum(dim=1)
    valid = pos > 0
    if not bool(valid.any()):
        return z.new_tensor(0.0)

    mean_log_prob_pos = (mask * log_prob).sum(dim=1) / (pos + 1e-12)
    loss = -mean_log_prob_pos[valid].mean()
    return loss


def _set_seed(seed: int) -> None:
    torch.manual_seed(int(seed))
    torch.cuda.manual_seed_all(int(seed))
    np.random.seed(int(seed))


@torch.inference_mode()
def _embed_loader(model: RetrievalEmbedder, loader: DataLoader, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    zs: list[torch.Tensor] = []
    ys: list[torch.Tensor] = []
    model.eval()
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        z = model(xb).float()
        zs.append(z.detach().cpu())
        ys.append(yb.detach().cpu())
    return torch.cat(zs, dim=0), torch.cat(ys, dim=0)


def _recall_at_k(z: torch.Tensor, y: torch.Tensor, ks: tuple[int, ...] = (1, 5)) -> dict[str, float]:
    z = F.normalize(z.float(), p=2, dim=1)
    y = y.to(dtype=torch.int64)
    n = int(z.shape[0])
    if n <= 1:
        return {f"r@{k}": 0.0 for k in ks}

    uniq, counts = torch.unique(y, return_counts=True)
    count_map = dict(zip(uniq.tolist(), counts.tolist(), strict=False))
    valid = torch.tensor([count_map[int(lbl)] > 1 for lbl in y.tolist()], dtype=torch.bool)
    if not bool(valid.any()):
        return {f"r@{k}": 0.0 for k in ks}

    z = z[valid]
    y = y[valid]
    n = int(z.shape[0])
    sim = z @ z.T
    sim.fill_diagonal_(-1e9)
    kmax = int(max(ks))
    idx = sim.topk(k=min(kmax, n - 1), dim=1).indices
    hits = y[idx] == y.view(-1, 1)
    out: dict[str, float] = {}
    for k in ks:
        kk = int(min(k, hits.shape[1]))
        out[f"r@{k}"] = float(hits[:, :kk].any(dim=1).float().mean().item())
    return out


def _save_checkpoint(path: Path, *, model: RetrievalEmbedder, cfg: RetrievalModelConfig, transform_spec, meta: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "model_cfg": cfg.__dict__,
            "state_dict": model.state_dict(),
            "transform_spec": transform_spec.__dict__,
            "meta": meta,
        },
        path,
    )


def train_retrieval_embedder(
    *,
    data_root: Path,
    labels_csv: Path,
    output_dir: Path,
    hf_endpoint: str | None,
    model_id: str = "facebook/dinov2-small",
    embed_dim: int = 256,
    freeze_backbone: bool = True,
    include_outliers: bool = True,
    datasets: list[str] | None = None,
    max_samples: int | None = None,
    epochs: int = 6,
    batch_size: int = 48,
    lr: float = 3e-4,
    weight_decay: float = 1e-4,
    temperature: float = 0.07,
    val_ratio: float = 0.1,
    seed: int = 42,
    num_workers: int = 4,
    fp16: bool = True,
    device: str | None = None,
    local_files_only: bool = False,
    cache_dir: Path | None = None,
) -> Path:
    if hf_endpoint:
        os.environ["HF_ENDPOINT"] = str(hf_endpoint)

    _set_seed(int(seed))

    ds_filter = set(datasets) if datasets else None
    samples, id_to_name = build_train_samples(
        data_root=data_root,
        labels_csv=labels_csv,
        datasets=ds_filter,
        include_outliers=bool(include_outliers),
    )
    if max_samples is not None:
        samples = samples[: int(max_samples)]

    train_samples, val_samples = split_train_val(samples, val_ratio=float(val_ratio), seed=int(seed))

    spec = load_transform_spec(
        model_id=str(model_id),
        hf_endpoint=str(hf_endpoint) if hf_endpoint else None,
        local_files_only=bool(local_files_only),
        cache_dir=cache_dir,
    )
    train_tf = make_train_transform(spec)
    eval_tf = make_eval_transform(spec)

    train_ds = RetrievalImageDataset(train_samples, transform=train_tf)
    val_ds = RetrievalImageDataset(val_samples, transform=eval_tf)

    dev = torch.device(device) if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[train] device={dev} model={model_id} samples={len(samples)} train={len(train_ds)} val={len(val_ds)}")

    cfg = RetrievalModelConfig(
        model_id=str(model_id),
        embed_dim=int(embed_dim),
        use_fp16=bool(fp16),
        local_files_only=bool(local_files_only),
        cache_dir=str(cache_dir) if cache_dir else None,
    )
    model = RetrievalEmbedder(cfg).to(dev)

    if bool(freeze_backbone):
        for p in model.backbone.parameters():
            p.requires_grad_(False)

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=float(lr), weight_decay=float(weight_decay))
    scaler = torch.amp.GradScaler("cuda", enabled=bool(fp16) and dev.type == "cuda")

    train_loader = DataLoader(
        train_ds,
        batch_size=int(batch_size),
        shuffle=True,
        num_workers=int(num_workers),
        pin_memory=(dev.type == "cuda"),
        drop_last=len(train_ds) >= int(batch_size),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=int(batch_size),
        shuffle=False,
        num_workers=int(num_workers),
        pin_memory=(dev.type == "cuda"),
        drop_last=False,
    )

    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "label_map.json").write_text(json.dumps(id_to_name, indent=2, default=str))

    meta = {
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "model_id": str(model_id),
        "embed_dim": int(embed_dim),
        "freeze_backbone": bool(freeze_backbone),
        "train_size": int(len(train_ds)),
        "val_size": int(len(val_ds)),
    }
    (output_dir / "train_config.json").write_text(json.dumps(meta, indent=2, default=str))

    best = -1.0
    for epoch in range(1, int(epochs) + 1):
        model.train()
        pbar = tqdm(train_loader, desc=f"epoch {epoch}/{epochs}", leave=False)
        losses: list[float] = []
        for xb, yb in pbar:
            xb = xb.to(dev, non_blocking=True)
            yb = yb.to(dev, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=bool(fp16) and dev.type == "cuda"):
                z = model(xb)
                loss = supervised_contrastive_loss(z, yb, temperature=float(temperature))
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            losses.append(float(loss.detach().cpu().item()))
            if losses:
                pbar.set_postfix(loss=f"{np.mean(losses):.4f}")

        metrics = {"epoch": epoch, "loss": float(np.mean(losses)) if losses else 0.0}
        if len(val_ds) > 1:
            z_val, y_val = _embed_loader(model, val_loader, dev)
            metrics.update(_recall_at_k(z_val, y_val, ks=(1, 5)))
        print(f"[train] epoch={epoch} " + " ".join([f"{k}={v:.4f}" for k, v in metrics.items() if k != "epoch"]))

        _save_checkpoint(output_dir / "last.pt", model=model, cfg=cfg, transform_spec=spec, meta=metrics)

        score = float(metrics.get("r@1", 0.0))
        if score >= best:
            best = score
            _save_checkpoint(output_dir / "best.pt", model=model, cfg=cfg, transform_spec=spec, meta=metrics)

    print(f"[train] done. best_r@1={best:.4f} checkpoint={output_dir/'best.pt'}")
    return output_dir / "best.pt"


# -------------------- Build retrieval cache --------------------


class _PathDataset(Dataset[Path]):
    def __init__(self, paths: list[Path], *, transform) -> None:
        self.paths = paths
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        p = self.paths[int(idx)]
        with Image.open(p) as im:
            im = im.convert("RGB")
            x = self.transform(im)
        return x, str(p)


def _load_checkpoint(path: Path) -> tuple[RetrievalModelConfig, TransformSpec, dict, dict]:
    ckpt = torch.load(path, map_location="cpu")
    model_cfg = RetrievalModelConfig(**ckpt["model_cfg"])
    spec = TransformSpec(**ckpt["transform_spec"])
    state = ckpt["state_dict"]
    meta = ckpt.get("meta", {})
    return model_cfg, spec, state, meta


def _make_pairs(emb: np.ndarray, *, k: int, mutual: bool) -> tuple[np.ndarray, np.ndarray]:
    emb = emb.astype(np.float32, copy=False)
    n = int(emb.shape[0])
    if n <= 1:
        return np.zeros((0,), np.int32), np.zeros((0,), np.int32)
    kk = int(min(k + 1, n))
    nnbr = NearestNeighbors(n_neighbors=kk, metric="cosine", algorithm="auto")
    nnbr.fit(emb)
    _dist, ind = nnbr.kneighbors(emb, return_distance=True)
    neigh = ind[:, 1:]

    directed: set[tuple[int, int]] = set()
    for i in range(n):
        for j in neigh[i].tolist():
            if i == int(j):
                continue
            directed.add((int(i), int(j)))

    if mutual:
        undirected: list[tuple[int, int]] = []
        for i, j in directed:
            if (j, i) in directed and i < j:
                undirected.append((i, j))
        undirected.sort()
        a = np.array([i for i, _ in undirected], dtype=np.int32)
        b = np.array([j for _, j in undirected], dtype=np.int32)
        return a, b

    edges = sorted(directed)
    a = np.array([i for i, _ in edges], dtype=np.int32)
    b = np.array([j for _, j in edges], dtype=np.int32)
    return a, b


@torch.inference_mode()
def _embed_paths_torch(
    model: nn.Module,
    paths: list[Path],
    *,
    transform,
    device: torch.device,
    batch_size: int,
    num_workers: int,
) -> np.ndarray:
    ds = _PathDataset(paths, transform=transform)
    loader = DataLoader(
        ds,
        batch_size=int(batch_size),
        shuffle=False,
        num_workers=int(num_workers),
        pin_memory=(device.type == "cuda"),
        drop_last=False,
    )
    zs: list[np.ndarray] = []
    for xb, _ in tqdm(loader, desc="embed", leave=False):
        xb = xb.to(device, non_blocking=True)
        z = model(xb).float().cpu().numpy()
        zs.append(z)
    return np.concatenate(zs, axis=0) if zs else np.zeros((0, 1), dtype=np.float32)


def _embed_paths_hist(paths: list[Path], *, bins: int = 16, resize: int = 64) -> np.ndarray:
    feats: list[np.ndarray] = []
    for p in tqdm(paths, desc="embed_hist", leave=False):
        with Image.open(p) as im:
            im = im.convert("RGB")
            if resize:
                im = im.resize((int(resize), int(resize)))
            arr = np.asarray(im).astype(np.float32) / 255.0
        hists = []
        for c in range(3):
            h, _ = np.histogram(arr[:, :, c], bins=int(bins), range=(0.0, 1.0), density=True)
            hists.append(h.astype(np.float32))
        v = np.concatenate(hists, axis=0)
        v = v / (np.linalg.norm(v) + 1e-12)
        feats.append(v)
    return np.stack(feats, axis=0) if feats else np.zeros((0, 3 * int(bins)), dtype=np.float32)


def build_retrieval_cache(
    *,
    data_root: Path,
    cache_root: Path,
    checkpoint: Path | None,
    model_id: str,
    hf_endpoint: str | None,
    datasets: list[str] | None = None,
    batch_size: int = 64,
    num_workers: int = 4,
    device: str | None = None,
    local_files_only: bool = False,
    cache_dir: Path | None = None,
    topk: int = 30,
    mutual: bool = True,
    overwrite: bool = False,
) -> None:
    if hf_endpoint:
        os.environ["HF_ENDPOINT"] = str(hf_endpoint)

    dev = torch.device(device) if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")

    use_ckpt = checkpoint is not None and Path(checkpoint).exists()

    backend = "pretrained"
    ckpt_meta: dict = {}
    ckpt_str: str | None = None

    model: nn.Module | None = None
    tf = None

    if use_ckpt:
        backend = "retrieval_finetune"
        ckpt_str = str(checkpoint)
        model_cfg, spec, state, ckpt_meta = _load_checkpoint(Path(checkpoint))

        # Allow overriding the backbone source (useful for offline Kaggle runs).
        model_cfg = RetrievalModelConfig(
            model_id=str(model_id),
            embed_dim=int(model_cfg.embed_dim),
            use_fp16=bool(model_cfg.use_fp16),
            local_files_only=bool(local_files_only),
            cache_dir=str(cache_dir) if cache_dir else None,
        )

        try:
            model = RetrievalEmbedder(model_cfg)
            model.load_state_dict(state, strict=True)
            model.eval().to(dev)
            tf = make_eval_transform(spec)
            embed_dim = int(model_cfg.embed_dim)
        except Exception as e:
            print(
                f"[warn] Could not load fine-tuned checkpoint backbone '{model_id}': {type(e).__name__}: {e}",
                "       Falling back to pretrained MODEL_REF (or histogram fallback).",
                flush=True,
            )
            backend = "pretrained"
            ckpt_str = None
            ckpt_meta = {}
            model = None
            tf = None
            use_ckpt = False

    if not use_ckpt:
        # Prefer pretrained MODEL_ID, but fall back to simple color-hist if Hugging Face weights are unavailable.
        try:
            spec = load_transform_spec(
                model_id=str(model_id),
                hf_endpoint=str(hf_endpoint) if hf_endpoint else None,
                local_files_only=bool(local_files_only),
                cache_dir=cache_dir,
            )
            tf = make_eval_transform(spec)

            backbone = _load_backbone(
                str(model_id),
                local_files_only=bool(local_files_only),
                cache_dir=cache_dir,
            )

            class _BackboneEmbedder(nn.Module):
                def __init__(self, backbone) -> None:
                    super().__init__()
                    self.backbone = backbone
                    self._dim = _backbone_dim(backbone)

                @property
                def embed_dim(self) -> int:
                    return int(self._dim)

                def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
                    out = self.backbone(pixel_values=pixel_values)
                    x = _pool_output(out)
                    z = F.normalize(x.float(), p=2, dim=1)
                    return z

            model = _BackboneEmbedder(backbone).to(dev)
            model.eval()
            embed_dim = int(model.embed_dim)
        except Exception as e:
            try:
                model, spec2, embed_dim, meta = _load_timm_dinov2_embedder(model_ref=str(model_id), device=dev)
                tf = make_eval_transform(spec2)
                backend = "timm_dinov2"
                ckpt_meta = meta
            except Exception as e2:
                print(
                    f"[warn] Could not load Hugging Face model '{model_id}': {type(e).__name__}: {e}",
                    f"       Could not load timm fallback either: {type(e2).__name__}: {e2}",
                    "       Falling back to simple color-hist embeddings (lower accuracy).",
                    flush=True,
                )
                backend = "histogram"
                model = None
                tf = None
                embed_dim = 48

    datasets_filter = set(datasets) if datasets else None
    cache_root.mkdir(parents=True, exist_ok=True)

    ds_dirs = [p for p in sorted(data_root.iterdir()) if p.is_dir()]
    for ds_dir in ds_dirs:
        dataset = ds_dir.name
        if datasets_filter and dataset not in datasets_filter:
            continue

        out_dir = cache_root / dataset
        out_dir.mkdir(parents=True, exist_ok=True)
        emb_path = out_dir / "embeddings.npy"
        pairs_path = out_dir / f"pairs_topk_K{int(topk)}.npz"
        meta_path = out_dir / "meta.json"

        if emb_path.exists() and pairs_path.exists() and meta_path.exists() and not overwrite:
            print(f"[skip] {dataset}: cache exists at {out_dir}")
            continue

        paths = _list_images(ds_dir)
        if not paths:
            print(f"[skip] {dataset}: no images in {ds_dir}")
            continue

        image_ids = [p.stem for p in paths]
        abs_paths = [p.resolve().as_posix() for p in paths]

        if backend == "histogram":
            emb = _embed_paths_hist(paths, bins=16, resize=64)
        else:
            assert model is not None and tf is not None
            emb = _embed_paths_torch(
                model,
                paths,
                transform=tf,
                device=dev,
                batch_size=int(batch_size),
                num_workers=int(num_workers),
            )

        a, b = _make_pairs(emb, k=int(topk), mutual=bool(mutual))
        np.save(emb_path, emb.astype(np.float32, copy=False))
        np.savez_compressed(pairs_path, a=a, b=b)

        meta_out = {
            "dataset": dataset,
            "backend": backend,
            "model": str(model_id) if backend != "histogram" else "histogram",
            "embed_dim": int(embed_dim),
            "checkpoint": ckpt_str,
            "image_ids": image_ids,
            "paths": abs_paths,
            "topk": int(topk),
            "mutual": bool(mutual),
            "checkpoint_meta": ckpt_meta,
        }
        meta_path.write_text(json.dumps(meta_out, indent=2))
        print(f"[ok] {dataset}: backend={backend} embeddings={emb.shape} pairs={len(a)} -> {out_dir}")


# -------------------- Cluster retrieval cache --------------------


def _normalize_rows(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float32, copy=False)
    denom = np.linalg.norm(x, axis=1, keepdims=True) + 1e-12
    return x / denom


def _knn_pairs(emb: np.ndarray, *, k: int, mutual: bool) -> list[tuple[int, int, float]]:
    emb = _normalize_rows(emb)
    n = int(emb.shape[0])
    if n <= 1:
        return []
    kk = int(min(k + 1, n))
    nnbr = NearestNeighbors(n_neighbors=kk, metric="cosine", algorithm="auto")
    nnbr.fit(emb)
    dist, ind = nnbr.kneighbors(emb, return_distance=True)
    neigh = ind[:, 1:]
    d = dist[:, 1:]
    sim = 1.0 - d

    directed: dict[tuple[int, int], float] = {}
    for i in range(n):
        for j, s in zip(neigh[i].tolist(), sim[i].tolist(), strict=False):
            if i == int(j):
                continue
            directed[(int(i), int(j))] = float(s)

    edges: list[tuple[int, int, float]] = []
    if mutual:
        for (i, j), w in directed.items():
            if (j, i) in directed and i < j:
                edges.append((i, j, max(float(w), float(directed[(j, i)]))))
        edges.sort()
        return edges

    for (i, j), w in directed.items():
        a, b = (i, j) if i < j else (j, i)
        edges.append((a, b, float(w)))
    edges.sort()
    return edges


class EdgeMLP(nn.Module):
    """Inference-only Edge MLP (optional)."""

    def __init__(self, d: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * int(d) + 1, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, zi, zj):
        cos = F.cosine_similarity(zi, zj).unsqueeze(1)
        x = torch.cat([zi * zj, torch.abs(zi - zj), cos], dim=1)
        return torch.sigmoid(self.net(x)).squeeze(1)


def _knn_pairs_scored(
    emb: np.ndarray,
    *,
    k: int,
    scorer: str,
    model,
    topm: int | None,
) -> list[tuple[int, int, float]]:
    emb = _normalize_rows(emb)
    n = emb.shape[0]
    if n <= 1:
        return []

    kk = min(k + 1, n)
    nnbr = NearestNeighbors(n_neighbors=kk, metric="cosine")
    nnbr.fit(emb)
    _, ind = nnbr.kneighbors(emb)
    neigh = ind[:, 1:]

    edges = []
    for i in range(n):
        js = neigh[i]
        zi = emb[i]
        zj = emb[js]

        if scorer == "mlp":
            with torch.no_grad():
                scores = model(
                    torch.from_numpy(np.repeat(zi[None, :], len(js), axis=0)),
                    torch.from_numpy(zj),
                ).numpy()

        elif scorer == "gbdt":
            cos = (zi * zj).sum(axis=1)
            X = np.stack([cos], axis=1)
            scores = model.predict_proba(X)[:, 1]

        else:
            raise ValueError(scorer)

        order = np.argsort(-scores)
        if topm is not None:
            order = order[: int(topm)]

        for o in order:
            j = int(js[o])
            if i < j:
                edges.append((i, j, float(scores[o])))

    edges.sort()
    return edges


def _cluster_graph(
    image_ids: list[str],
    edges: list[tuple[int, int, float]],
    *,
    min_sim: float,
    method: str,
    seed: int,
) -> list[list[str]]:
    g = nx.Graph()
    g.add_nodes_from(image_ids)
    for i, j, w in edges:
        if float(w) >= float(min_sim):
            g.add_edge(image_ids[int(i)], image_ids[int(j)], weight=float(w))

    if g.number_of_edges() == 0:
        return [[n] for n in image_ids]

    method = method.lower().strip()
    if method == "louvain" and hasattr(nx.algorithms.community, "louvain_communities"):
        comms = list(nx.algorithms.community.louvain_communities(g, weight="weight", seed=int(seed)))
    elif method in {"greedy", "greedy_modularity"}:
        comms = list(nx.algorithms.community.greedy_modularity_communities(g, weight="weight"))
    else:
        comms = [set(c) for c in nx.connected_components(g)]

    comms = [sorted([str(x) for x in c]) for c in comms]
    comms.sort(key=lambda c: (-len(c), c[0] if c else ""))
    return comms


def _label_clusters(dataset: str, image_ids: list[str], clusters: list[list[str]], *, min_cluster_size: int) -> pd.DataFrame:
    scene_of: dict[str, str] = {}
    idx = 1
    for c in clusters:
        if len(c) < int(min_cluster_size):
            continue
        scene = f"cluster_{idx:04d}"
        idx += 1
        for im in c:
            scene_of[str(im)] = scene
    for im in image_ids:
        scene_of.setdefault(str(im), "outliers")
    df = pd.DataFrame({"dataset": dataset, "image_id": image_ids})
    df["scene"] = df["image_id"].map(scene_of)
    return df


def cluster_retrieval_cache(
    *,
    cache_root: Path,
    out_csv: Path,
    datasets: list[str] | None = None,
    topk: int = 30,
    mutual: bool = True,
    min_sim: float = 0.35,
    method: str = "louvain",
    min_cluster_size: int = 3,
    seed: int = 42,
    edge_scorer: str = "cosine",
    edge_model: Path | None = None,
    edge_topm: int | None = None,
) -> None:
    if not cache_root.exists():
        raise FileNotFoundError(f"cache-root not found: {cache_root}")

    datasets_filter = set(datasets) if datasets else None

    edge_scorer = str(edge_scorer).lower().strip()
    scorer_model = None
    if edge_scorer in {"mlp", "gbdt"}:
        if edge_model is None:
            raise ValueError(f"edge_scorer={edge_scorer} requires edge_model")
        if edge_scorer == "mlp":
            ckpt = torch.load(edge_model, map_location="cpu")
            scorer_model = EdgeMLP(int(ckpt["dim"]))
            scorer_model.load_state_dict(ckpt["state_dict"])  # type: ignore[arg-type]
            scorer_model.eval()
        else:
            scorer_model = joblib.load(edge_model)

    out_rows: list[pd.DataFrame] = []
    for ds_dir in sorted([p for p in cache_root.iterdir() if p.is_dir() and (p / "meta.json").exists()]):
        dataset = ds_dir.name
        if datasets_filter and dataset not in datasets_filter:
            continue
        emb_path = ds_dir / "embeddings.npy"
        meta_path = ds_dir / "meta.json"
        if not emb_path.exists() or not meta_path.exists():
            continue
        meta = json.loads(meta_path.read_text())
        image_ids = [str(x) for x in meta.get("image_ids", [])]
        emb = np.load(emb_path)

        if len(image_ids) != int(emb.shape[0]):
            raise ValueError(f"{dataset}: meta/image_ids mismatch embeddings rows")

        if edge_scorer == "cosine":
            edges = _knn_pairs(emb, k=int(topk), mutual=bool(mutual))
        else:
            edges = _knn_pairs_scored(
                emb,
                k=int(topk),
                scorer=edge_scorer,
                model=scorer_model,
                topm=edge_topm,
            )

        clusters = _cluster_graph(image_ids, edges, min_sim=float(min_sim), method=str(method), seed=int(seed))
        df = _label_clusters(dataset, image_ids, clusters, min_cluster_size=int(min_cluster_size))
        out_rows.append(df)
        n_clusters = int(df.loc[df["scene"] != "outliers", "scene"].nunique())
        n_out = int((df["scene"] == "outliers").sum())
        print(f"[ok] {dataset}: clusters={n_clusters} outliers={n_out} images={len(df)}")

    if not out_rows:
        raise RuntimeError("No datasets found in cache-root (missing meta.json/embeddings.npy)")

    out = pd.concat(out_rows, ignore_index=True)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(out_csv, index=False)
    print(f"[ok] wrote {out_csv} rows={len(out)}")


# -------------------- COLMAP sparse SfM per cluster --------------------


def _read_clusters_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = {"dataset", "image_id", "scene"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path} is missing columns: {sorted(missing)}")
    df = df.copy()
    df["dataset"] = df["dataset"].astype(str)
    df["scene"] = df["scene"].astype(str)
    df["image_id"] = df["image_id"].astype(str)
    return df


def _load_meta_map(cache_root: Path, dataset: str) -> dict[str, Path]:
    meta_path = cache_root / dataset / "meta.json"
    if not meta_path.exists():
        return {}
    meta = json.loads(meta_path.read_text())
    image_ids = meta.get("image_ids", [])
    paths = meta.get("paths", [])
    if not isinstance(image_ids, list) or not isinstance(paths, list) or len(image_ids) != len(paths):
        return {}
    out: dict[str, Path] = {}
    for image_id, p in zip(image_ids, paths, strict=False):
        out[str(image_id)] = Path(str(p))
    return out


def _resolve_image_path(
    *,
    data_root: Path,
    dataset: str,
    image_id: str,
    meta_map: dict[str, Path],
) -> Path | None:
    if image_id in meta_map:
        p = meta_map[image_id]
        if p.is_absolute() and p.exists():
            return p
        if p.exists():
            return p.resolve()

    base = data_root / dataset / image_id
    for ext in (".png", ".jpg", ".jpeg", ".webp"):
        p = base.with_suffix(ext)
        if p.exists():
            return p
    return None


def _copy(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.is_file():
        return
    if dst.exists() and dst.is_symlink():
        dst.unlink()
    shutil.copy2(src, dst)


def run_colmap_sfm_clusters(
    *,
    clusters_csv: Path,
    data_root: Path,
    cache_root: Path,
    output_root: Path,
    runner: str = "local",
    matcher: str = "colmap",
    outliers_label: str = "outliers",
    min_images: int = 4,
    overwrite: bool = False,
    camera_model: str = "SIMPLE_RADIAL",
    single_camera: bool = True,
    mapper_min_model_size: int = 3,
    mapper_init_min_num_inliers: int = 15,
    mapper_min_num_matches: int | None = None,
    mapper_init_min_tri_angle: float = 2.0,
    mapper_tri_min_angle: float = 1.5,
    mapper_filter_min_tri_angle: float = 1.5,
    mapper_tri_ignore_two_view_tracks: int = 0,
    sift_use_gpu: bool = False,
) -> int:
    if runner != "local":
        raise ValueError("This standalone notebook only supports RUNNER='local' (Docker is not available on Kaggle).")
    if matcher != "colmap":
        raise ValueError("This standalone notebook only supports MATCHER='colmap'.")
    resolved_colmap = _resolve_colmap_bin()
    if not resolved_colmap:
        print("[debug] IMC25_COLMAP_BIN:", os.environ.get("IMC25_COLMAP_BIN"), flush=True)
        print("[debug] which colmap:", shutil.which("colmap"), flush=True)
        if Path("/kaggle/input").exists():
            cmd = "find /kaggle/input -maxdepth 8 -type f -name colmap | head -n 20"
            print("+", cmd, flush=True)
            subprocess.run(["bash", "-lc", cmd], check=False)
        raise FileNotFoundError(
            "`colmap` not found. Add a COLMAP binary via Kaggle 'Add Dataset/Model' and set IMC25_COLMAP_BIN to the executable (or folder containing it)."
        )

    df = _read_clusters_csv(clusters_csv)
    ok_scenes = 0
    attempted_scenes = 0
    skipped_scenes = 0
    failed_scenes = 0

    for (dataset, scene), grp in df.groupby(["dataset", "scene"], sort=True):
        if str(scene).lower() == str(outliers_label).lower():
            continue
        image_ids = grp["image_id"].astype(str).tolist()
        if len(image_ids) < int(min_images):
            skipped_scenes += 1
            continue

        out_dir = output_root / f"{dataset}_{scene}"
        images_dir = out_dir / "images"
        sparse_root = out_dir / "sparse"
        db_path = out_dir / "colmap.db"

        if sparse_root.exists() and any((p / "images.bin").exists() for p in sparse_root.iterdir() if p.is_dir()) and not overwrite:
            print(f"[skip] {dataset}/{scene}: sparse model exists")
            ok_scenes += 1
            continue

        if overwrite and out_dir.exists():
            shutil.rmtree(out_dir)

        images_dir.mkdir(parents=True, exist_ok=True)
        sparse_root.mkdir(parents=True, exist_ok=True)

        meta_map = _load_meta_map(cache_root, str(dataset))
        resolved: list[Path] = []
        for image_id in image_ids:
            p = _resolve_image_path(data_root=data_root, dataset=str(dataset), image_id=str(image_id), meta_map=meta_map)
            if p is None:
                continue
            resolved.append(p)

        if len(resolved) < 2:
            print(f"[skip] {dataset}/{scene}: not enough images found ({len(resolved)})")
            skipped_scenes += 1
            continue

        desired_names = {p.name for p in resolved}
        for entry in list(images_dir.iterdir()):
            if entry.is_file() and entry.name not in desired_names:
                entry.unlink(missing_ok=True)
        for src in resolved:
            _copy(src, images_dir / src.name)

        if db_path.exists():
            db_path.unlink(missing_ok=True)

        feature_args = [
            "feature_extractor",
            "--database_path",
            str(db_path),
            "--image_path",
            str(images_dir),
            "--ImageReader.camera_model",
            str(camera_model),
        ]
        if single_camera:
            feature_args += ["--ImageReader.single_camera", "1"]
        feature_args += ["--SiftExtraction.use_gpu", "1" if sift_use_gpu else "0"]

        match_args = ["exhaustive_matcher", "--database_path", str(db_path)]
        match_args += ["--SiftMatching.use_gpu", "1" if sift_use_gpu else "0"]

        mapper_args = [
            "mapper",
            "--database_path",
            str(db_path),
            "--image_path",
            str(images_dir),
            "--output_path",
            str(sparse_root),
            "--Mapper.ba_refine_principal_point",
            "0",
            "--Mapper.min_model_size",
            str(int(mapper_min_model_size)),
            "--Mapper.init_min_num_inliers",
            str(int(mapper_init_min_num_inliers)),
            "--Mapper.init_min_tri_angle",
            str(float(mapper_init_min_tri_angle)),
            "--Mapper.tri_min_angle",
            str(float(mapper_tri_min_angle)),
            "--Mapper.filter_min_tri_angle",
            str(float(mapper_filter_min_tri_angle)),
            "--Mapper.tri_ignore_two_view_tracks",
            str(int(mapper_tri_ignore_two_view_tracks)),
        ]
        if mapper_min_num_matches is None:
            mapper_min_num_matches = int(mapper_init_min_num_inliers)
        mapper_args += ["--Mapper.min_num_matches", str(int(mapper_min_num_matches))]

        attempted_scenes += 1
        print(f"[run] {dataset}/{scene}: images={len(resolved)}", flush=True)

        try:
            run_cmd(["colmap", *feature_args])
            run_cmd(["colmap", *match_args])
        except subprocess.CalledProcessError as e:
            print(f"[warn] {dataset}/{scene}: feature/match failed; skipping (rc={e.returncode})", flush=True)
            if getattr(e, "output", None):
                print(str(e.output), flush=True)
            failed_scenes += 1
            continue

        def _replace_arg(args: list[str], flag: str, value: str) -> list[str]:
            out = args[:]
            for i in range(len(out) - 1):
                if out[i] == flag:
                    out[i + 1] = value
                    return out
            return out + [flag, value]

        attempts = [
            ("default", int(mapper_init_min_num_inliers), float(mapper_init_min_tri_angle), float(mapper_tri_min_angle)),
            ("relaxed1", 8, 0.5, 0.5),
            ("relaxed2", 4, 0.1, 0.1),
        ]

        ok = False
        for label, min_inliers, init_tri_angle, tri_min_angle in attempts:
            # Clean sparse_root between attempts to avoid mixing partial outputs.
            if sparse_root.exists():
                shutil.rmtree(sparse_root)
            sparse_root.mkdir(parents=True, exist_ok=True)

            args_try = mapper_args[:]
            args_try = _replace_arg(args_try, "--Mapper.init_min_num_inliers", str(int(min_inliers)))
            args_try = _replace_arg(args_try, "--Mapper.min_num_matches", str(int(min_inliers)))
            args_try = _replace_arg(args_try, "--Mapper.init_min_tri_angle", str(float(init_tri_angle)))
            args_try = _replace_arg(args_try, "--Mapper.tri_min_angle", str(float(tri_min_angle)))
            args_try = _replace_arg(args_try, "--Mapper.filter_min_tri_angle", str(float(tri_min_angle)))

            try:
                run_cmd(["colmap", *args_try])
                ok = True
                break
            except subprocess.CalledProcessError as e:
                print(f"[warn] {dataset}/{scene}: mapper failed ({label}) rc={e.returncode}", flush=True)
                if getattr(e, "output", None):
                    print(str(e.output), flush=True)
                ok = False

        if not ok:
            print(f"[warn] {dataset}/{scene}: mapper failed for all attempts; poses will be NaN.", flush=True)
            failed_scenes += 1
            continue

        ok_scenes += 1
        continue

    print(
        f"[colmap] scenes_ok={ok_scenes} attempted={attempted_scenes} skipped={skipped_scenes} failed={failed_scenes}",
        flush=True,
    )
    return int(ok_scenes)


## 1) Train retrieval embedder

This uses the training labels to learn an embedding space where images from the same scene are close.


In [ ]:
if DO_TRAIN:
    TRAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINT = train_retrieval_embedder(
        data_root=TRAIN_ROOT,
        labels_csv=TRAIN_LABELS_CSV,
        output_dir=TRAIN_OUTPUT_DIR,
        hf_endpoint=os.environ.get("HF_ENDPOINT"),
        model_id=MODEL_REF,
        num_workers=0,
        local_files_only=HF_LOCAL_ONLY,
        cache_dir=HF_CACHE_DIR,
    )

if CHECKPOINT is not None and Path(CHECKPOINT).exists():
    print("checkpoint:", CHECKPOINT)
else:
    print(
        "[info] Fine-tuned checkpoint not found; will embed test images using MODEL_REF (offline).",
        flush=True,
    )
    CHECKPOINT = None


## 2) Embed test images → `cache_retrieval/`


In [ ]:
if DO_EMBED_TEST:
    build_retrieval_cache(
        data_root=TEST_ROOT,
        cache_root=CACHE_RETRIEVAL,
        checkpoint=CHECKPOINT,
        model_id=MODEL_REF,
        hf_endpoint=os.environ.get("HF_ENDPOINT"),
        local_files_only=HF_LOCAL_ONLY,
        cache_dir=HF_CACHE_DIR,
        topk=30,
        mutual=True,
        num_workers=0,
    )

assert CACHE_RETRIEVAL.exists(), f"Missing {CACHE_RETRIEVAL}"
sorted([p.name for p in CACHE_RETRIEVAL.iterdir() if p.is_dir()])[:10]


## 3) Cluster embeddings → `cache/clusters_retrieval.csv`


In [12]:
if DO_CLUSTER:
    if EDGE_SCORER in {"mlp", "gbdt"}:
        if EDGE_MODEL is None or not EDGE_MODEL.exists():
            raise FileNotFoundError(f"Missing edge model: {EDGE_MODEL}")

    cluster_retrieval_cache(
        cache_root=CACHE_RETRIEVAL,
        out_csv=CLUSTERS_CSV,
        topk=30,
        mutual=True,
        edge_scorer=EDGE_SCORER,
        edge_model=EDGE_MODEL,
        edge_topm=int(EDGE_TOPM) if EDGE_TOPM is not None else None,
    )

assert CLUSTERS_CSV.exists(), f"Missing {CLUSTERS_CSV}"
pd.read_csv(CLUSTERS_CSV).head()


+ /Users/luca/Documents/Uni/Tsinghua/Machine Learning/ImageMatchingChallenge2025/.venv/bin/python scripts/cluster_retrieval.py --cache-root /Users/luca/Documents/Uni/Tsinghua/Machine Learning/ImageMatchingChallenge2025/cache_retrieval --out-csv /Users/luca/Documents/Uni/Tsinghua/Machine Learning/ImageMatchingChallenge2025/cache/clusters_retrieval.csv --edge-scorer cosine
Inference embeddings shape: (22, 256)
[ok] ETs: clusters=2 outliers=0 images=22
Inference embeddings shape: (51, 256)
[ok] stairs: clusters=3 outliers=0 images=51
[ok] wrote /Users/luca/Documents/Uni/Tsinghua/Machine Learning/ImageMatchingChallenge2025/cache/clusters_retrieval.csv rows=73


,dataset,image_id,scene
0,ETs,another_et_another_et001,cluster_0001
1,ETs,another_et_another_et002,cluster_0001
2,ETs,another_et_another_et003,cluster_0001
3,ETs,another_et_another_et004,cluster_0001
4,ETs,another_et_another_et005,cluster_0001


## 4) Run SfM per cluster (COLMAP)

This produces sparse reconstructions under:

`outputs_submission/<dataset>_<scene>/sparse/<model_id>/images.bin`

By default we use `SPARSE_ONLY=True` (submission does not require dense point clouds).


In [ ]:
if DO_COLMAP_SFM:
    if RUNNER != "local":
        raise ValueError("Kaggle does not support Docker; set RUNNER='local'.")
    if MATCHER != "colmap":
        raise ValueError("This notebook only supports MATCHER='colmap'.")

    ok_scenes = run_colmap_sfm_clusters(
        clusters_csv=CLUSTERS_CSV,
        data_root=TEST_ROOT,
        cache_root=CACHE_RETRIEVAL,
        output_root=OUTPUT_ROOT,
        runner=RUNNER,
        matcher=MATCHER,
        overwrite=False,
        sift_use_gpu=bool(COLMAP_SIFT_USE_GPU),
    )

    if int(ok_scenes) == 0:
        cmd = f"find {str(OUTPUT_ROOT)} -maxdepth 5 -type f -name images.bin | head -n 20"
        print("+", cmd, flush=True)
        subprocess.run(["bash", "-lc", cmd], check=False)
        raise RuntimeError(
            "COLMAP produced zero sparse models (no images.bin). Refusing to export an all-NaN submission. "
            "Fix COLMAP (set IMC25_COLMAP_BIN) and rerun SfM."
        )


## 5) Export `submission.csv`

We read:
- cluster assignment from `cache/clusters_retrieval.csv`
- COLMAP poses from `outputs_submission/<dataset>_<scene>/sparse/.../images.bin`

For images that are outliers or were not registered by SfM, we write `nan` poses.


In [ ]:
IMG_EXTS = (".png", ".jpg", ".jpeg", ".webp")


def qvec2rotmat(q: np.ndarray) -> np.ndarray:
    # COLMAP quaternion format: [qw, qx, qy, qz]
    w, x, y, z = [float(v) for v in q.tolist()]
    return np.array(
        [
            [1 - 2 * y * y - 2 * z * z, 2 * x * y - 2 * w * z, 2 * x * z + 2 * w * y],
            [2 * x * y + 2 * w * z, 1 - 2 * x * x - 2 * z * z, 2 * y * z - 2 * w * x],
            [2 * x * z - 2 * w * y, 2 * y * z + 2 * w * x, 1 - 2 * x * x - 2 * y * y],
        ],
        dtype=np.float64,
    )


def read_images_bin(path: Path) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    """Return map: image_name -> (qvec, tvec)."""
    out: dict[str, tuple[np.ndarray, np.ndarray]] = {}
    with path.open("rb") as f:
        num_images = struct.unpack("<Q", f.read(8))[0]
        for _ in range(int(num_images)):
            _image_id = struct.unpack("<i", f.read(4))[0]
            qvec = np.array(struct.unpack("<4d", f.read(32)), dtype=np.float64)
            tvec = np.array(struct.unpack("<3d", f.read(24)), dtype=np.float64)
            _camera_id = struct.unpack("<i", f.read(4))[0]

            name_bytes = bytearray()
            while True:
                c = f.read(1)
                if c == b"\x00" or c == b"":
                    break
                name_bytes.extend(c)
            name = name_bytes.decode("utf-8")

            num_points2d = struct.unpack("<Q", f.read(8))[0]
            f.seek(int(num_points2d) * (2 * 8 + 8), 1)

            out[name] = (qvec, tvec)
    return out


def colmap_bin_count(path: Path) -> int:
    try:
        with path.open("rb") as f:
            head = f.read(8)
        if len(head) != 8:
            return 0
        return int(struct.unpack("<Q", head)[0])
    except Exception:
        return 0


def pick_best_sparse_model(sparse_root: Path) -> Path | None:
    if not sparse_root.exists():
        return None
    candidates = [p for p in sparse_root.iterdir() if p.is_dir() and p.name.isdigit()]
    if not candidates:
        return None
    best = None
    best_path = None
    for p in sorted(candidates, key=lambda x: int(x.name)):
        n_images = colmap_bin_count(p / "images.bin")
        n_points = colmap_bin_count(p / "points3D.bin")
        score = (n_images, n_points, p.name)
        if best is None or score > best:
            best = score
            best_path = p
    return best_path


def load_meta_map(cache_root: Path, dataset: str) -> dict[str, Path]:
    meta_path = cache_root / dataset / "meta.json"
    if not meta_path.exists():
        return {}
    meta = json.loads(meta_path.read_text())
    image_ids = meta.get("image_ids", [])
    paths = meta.get("paths", [])
    if not isinstance(image_ids, list) or not isinstance(paths, list) or len(image_ids) != len(paths):
        return {}
    out: dict[str, Path] = {}
    for image_id, p in zip(image_ids, paths, strict=False):
        pp = Path(str(p))
        pp = pp if pp.is_absolute() else (WORK_ROOT / pp)
        out[str(image_id)] = pp.resolve()
    return out


def resolve_image_name(*, dataset: str, image_id: str, meta_map: dict[str, Path]) -> str:
    if image_id in meta_map:
        return meta_map[image_id].name
    # fallback: try <stem>.<ext>
    for ext in IMG_EXTS:
        p = TEST_ROOT / dataset / f"{image_id}{ext}"
        if p.exists():
            return p.name
    # last resort: assume image_id is already a filename
    return image_id


def fmt_mat9(R: np.ndarray) -> str:
    v = R.reshape(-1).tolist()
    return ";".join([f"{float(x):.9f}" for x in v])


def fmt_vec3(t: np.ndarray) -> str:
    v = t.reshape(-1).tolist()
    return ";".join([f"{float(x):.9f}" for x in v])


In [ ]:
clusters = pd.read_csv(CLUSTERS_CSV)
assert {"dataset", "scene", "image_id"}.issubset(set(clusters.columns))

sample_csv = DATA_ROOT / "sample_submission.csv"
if not sample_csv.exists():
    raise FileNotFoundError(f"Missing sample_submission.csv at {sample_csv} (needed to get image_id values).")
template = pd.read_csv(sample_csv)
required_cols = {"image_id", "dataset", "scene", "image"}
missing = required_cols - set(template.columns)
if missing:
    raise ValueError(f"sample_submission.csv missing columns: {sorted(missing)}")

meta_maps: dict[str, dict[str, Path]] = {}
all_datasets = set(template["dataset"].astype(str).unique().tolist()) | set(clusters["dataset"].astype(str).unique().tolist())
for ds in sorted(all_datasets):
    meta_maps[ds] = load_meta_map(CACHE_RETRIEVAL, ds)

# Build lookup: (dataset, image filename) -> predicted cluster label
cluster_lookup: dict[tuple[str, str], str] = {}
for r in clusters.itertuples(index=False):
    ds = str(r.dataset)
    scene = str(r.scene)
    image_id = str(r.image_id)
    meta_map = meta_maps.get(ds, {})
    image_name = resolve_image_name(dataset=ds, image_id=image_id, meta_map=meta_map)
    cluster_lookup[(ds, image_name)] = scene

# Load COLMAP poses per (dataset, scene)
pose_maps: dict[tuple[str, str], dict[str, tuple[np.ndarray, np.ndarray]]] = {}
for (ds, scene), grp in clusters.groupby(["dataset", "scene"], sort=True):
    if str(scene).lower() == "outliers":
        continue
    out_dir = OUTPUT_ROOT / f"{ds}_{scene}"
    sparse_root = out_dir / "sparse"
    best = pick_best_sparse_model(sparse_root)
    if best is None:
        pose_maps[(str(ds), str(scene))] = {}
        continue
    images_bin = best / "images.bin"
    if not images_bin.exists():
        pose_maps[(str(ds), str(scene))] = {}
        continue
    pose_maps[(str(ds), str(scene))] = read_images_bin(images_bin)

rows = []
for r in template.itertuples(index=False):
    ds = str(r.dataset)
    image_name = str(r.image)
    submission_image_id = str(r.image_id)

    scene = cluster_lookup.get((ds, image_name), "outliers")
    if str(scene).lower() == "outliers":
        rows.append(
            {
                "image_id": submission_image_id,
                "dataset": ds,
                "scene": str(scene),
                "image": image_name,
                "rotation_matrix": "nan;nan;nan;nan;nan;nan;nan;nan;nan",
                "translation_vector": "nan;nan;nan",
            }
        )
        continue

    pose_map = pose_maps.get((ds, str(scene)), {})
    pose = pose_map.get(image_name)
    if pose is None:
        rows.append(
            {
                "image_id": submission_image_id,
                "dataset": ds,
                "scene": str(scene),
                "image": image_name,
                "rotation_matrix": "nan;nan;nan;nan;nan;nan;nan;nan;nan",
                "translation_vector": "nan;nan;nan",
            }
        )
        continue

    qvec, tvec = pose
    R = qvec2rotmat(qvec)
    rows.append(
        {
            "image_id": submission_image_id,
            "dataset": ds,
            "scene": str(scene),
            "image": image_name,
            "rotation_matrix": fmt_mat9(R),
            "translation_vector": fmt_vec3(tvec),
        }
    )

sub = pd.DataFrame(
    rows,
    columns=["image_id", "dataset", "scene", "image", "rotation_matrix", "translation_vector"],
)
sub.to_csv(SUBMISSION_CSV, index=False)
try:
    out_copy = OUTPUT_ROOT / "submission.csv"
    if out_copy.resolve() != Path(SUBMISSION_CSV).resolve():
        shutil.copy2(SUBMISSION_CSV, out_copy)
        print("also wrote", out_copy)
except Exception as e:
    print("[warn] could not copy submission into OUTPUT_ROOT:", e)
print("wrote", SUBMISSION_CSV, "rows=", len(sub))
sub.head()


## Sanity checks


In [ ]:
sub = pd.read_csv(SUBMISSION_CSV)
print("rows:", len(sub))
print("datasets:", sub["dataset"].nunique())
print("scenes:", sub["scene"].nunique())
is_nan_pose = sub["translation_vector"].astype(str).str.contains("nan")
print("nan poses:", int(is_nan_pose.sum()), "/", len(sub))
sub.head(10)
